In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv gymnasium


# MLFP05 — Exercise 8.1: Deep Q-Network (DQN) from Scratch


**WHAT YOU'LL LEARN:**
After completing this exercise, you will be able to:
- Explain Q-learning intuition: "learning the value of actions by
trial and error"
- Implement a replay buffer ("learning from memories") and explain
why it breaks temporal correlation
- Implement a target network ("aiming at a moving target") and explain
why it stabilises the Bellman update
- Implement epsilon-greedy exploration ("explore vs exploit") and
watch the agent shift from random to purposeful action
- Train DQN on CartPole-v1 and track all metrics with ExperimentTracker
- Visualise reward curves, epsilon decay, Q-value heatmaps, and
episode length progression
- Apply DQN to a Singapore retail inventory management problem


**PREREQUISITES: M5/ex_2 through M5/ex_4 (PyTorch training loops).**

**ESTIMATED TIME: ~40 min**
DATASETS: No static dataset — the environment IS the data source.
- CartPole-v1 (Gymnasium classic control, 4-D state, 2 actions)


**TASKS:**
1. Understand DQN theory and the Bellman equation
2. Build DQN: replay buffer, Q-network, training loop
3. Train DQN on CartPole-v1 with ExperimentTracker logging
4. Visualise agent behaviour: reward curve, epsilon decay, Q-value
heatmap, episode lengths
5. Apply: inventory management for a Singapore retailer



In [ ]:
from __future__ import annotations

import asyncio
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

import gymnasium as gym
import polars as pl

from shared.mlfp05.ex_8 import (
    DQN,
    OUTPUT_DIR,
    ReplayBuffer,
    device,
    evaluate_policy,
    make_cartpole,
    moving_average,
    register_rl_model,
    setup_engines,
)
from kailash_ml import ModelVisualizer


## TASK 1 — Theory: Why DQN Exists
Imagine you're a new hire at a Singapore grocery chain. Every day you
make decisions — should you restock the shelves? Run a promotion?
Rearrange the display? You don't know the "right" answers yet, so you
TRY things and observe the RESULTS (sales go up, complaints go down).
Over time, you learn which actions are valuable in which situations.
That's Q-learning. Q(s, a) answers: "If I'm in situation s and I take
action a, how much total future reward can I expect?"
The BELLMAN OPTIMALITY EQUATION is the recursive definition:
Q*(s, a) = E[ r + gamma * max_{a'} Q*(s', a') ]
"The value of an action = immediate reward + best future value"
DQN (Mnih et al., 2015) approximates Q* with a neural network. Two
innovations prevent training instability:
(1) REPLAY BUFFER — "Learning from memories"
Problem: consecutive (s, a, r, s') samples are correlated — the
agent sees similar states in sequence, biasing gradient updates.
Solution: store transitions in a buffer, sample RANDOM minibatches.
Like a student reviewing flashcards in random order instead of
re-reading the textbook front to back.
(2) TARGET NETWORK — "Aiming at a moving target"
Problem: Q appears on BOTH sides of the Bellman equation. Updating
Q to match a target that is also Q creates oscillations — you're
chasing your own tail.
Solution: keep a SLOW-MOVING COPY (target network) for computing
targets. Update it periodically, not every step. Like calibrating
a scale against a reference weight, not against itself.
(3) EPSILON-GREEDY — "Explore vs exploit"
With probability epsilon, take a RANDOM action (explore new
strategies). Otherwise, take the BEST action according to Q
(exploit what you've learned). Epsilon starts high (mostly random)
and decays over training (mostly purposeful).


In [ ]:
print("=" * 70)
print("  TASK 1: DQN Theory — Q-Learning with Neural Networks")
print("=" * 70)


## TASK 2 — Build DQN: training loop with replay buffer + target network


In [ ]:
# Set up CartPole and kailash engines
cartpole_env, obs_dim, n_actions = make_cartpole()
conn, tracker, exp_name, registry, has_registry = setup_engines()


async def train_dqn_async(
    env: gym.Env,
    obs_d: int,
    n_act: int,
    n_episodes: int = 200,
    gamma: float = 0.99,
    lr: float = 1e-3,
    batch_size: int = 64,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.01,
    epsilon_decay: float = 0.995,
    target_update_freq: int = 10,
    min_replay_size: int = 500,
    run_name: str = "dqn_cartpole",
) -> tuple[DQN, list[float], list[float], list[float], list[int]]:
    """Train DQN and log to ExperimentTracker.

    Returns:
        (q_net, episode_rewards, episode_losses, epsilons, episode_lengths)
    """
    # TODO: Create q_net (DQN) and target_net (DQN copy) on device
    # Hint: target_net starts with the same weights as q_net via load_state_dict
    q_net = # TODO
    target_net = # TODO
    # TODO: Set target_net to eval mode and copy q_net weights
    # Hint: target_net.load_state_dict(q_net.state_dict()); target_net.eval()

    optimizer = torch.optim.Adam(q_net.parameters(), lr=lr)
    replay = ReplayBuffer(capacity=10_000)
    epsilon = epsilon_start

    episode_rewards: list[float] = []
    episode_losses: list[float] = []
    epsilons: list[float] = []
    episode_lengths: list[int] = []

    async with tracker.run(experiment_name=exp_name, run_name=run_name) as ctx:
        await ctx.log_params(
            {
                "algorithm": "DQN",
                "gamma": str(gamma),
                "lr": str(lr),
                "epsilon_start": str(epsilon_start),
                "epsilon_end": str(epsilon_end),
                "target_update_freq": str(target_update_freq),
                "batch_size": str(batch_size),
            }
        )

        print(f"\n== Training DQN: {run_name} ==")
        for ep in range(n_episodes):
            state, _ = env.reset(seed=42 + ep)
            total_reward = 0.0
            ep_loss_sum = 0.0
            ep_loss_count = 0
            steps = 0
            done = False

            while not done:
                # TODO: Epsilon-greedy action selection
                # Hint: with probability epsilon choose random action
                # (env.action_space.sample()), otherwise argmax of Q-values
                # from q_net on the current state tensor
                if random.random() < epsilon:
                    action = # TODO: random action
                else:
                    with torch.no_grad():
                        s_t = torch.tensor(state, dtype=torch.float32, device=device)
                        action = # TODO: argmax of q_net(s_t)

                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                replay.push(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward
                steps += 1

                # Train on a minibatch from replay buffer
                if len(replay) >= min_replay_size:
                    s_b, a_b, r_b, ns_b, d_b = replay.sample(batch_size)

                    # TODO: Compute current Q-values for chosen actions
                    # Hint: q_net(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)
                    q_values = # TODO

                    # TODO: Compute target Q-values using Bellman equation
                    # Target: r + gamma * max_a' Q_target(s', a') * (1 - done)
                    # Hint: use target_net (not q_net) for next-state Q-values
                    with torch.no_grad():
                        next_q = # TODO: target_net(ns_b).max(dim=1).values
                        targets = # TODO: r_b + gamma * next_q * (1.0 - d_b)

                    # TODO: Compute MSE loss between q_values and targets
                    # Hint: F.mse_loss(q_values, targets)
                    loss = # TODO
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    ep_loss_sum += loss.item()
                    ep_loss_count += 1

            # TODO: Decay epsilon after each episode
            # Hint: epsilon = max(epsilon_end, epsilon * epsilon_decay)
            epsilon = # TODO

            # TODO: Update target network periodically
            # Hint: every target_update_freq episodes, copy q_net weights to target_net
            if (ep + 1) % target_update_freq == 0:
                # TODO: target_net.load_state_dict(q_net.state_dict())
                pass

            episode_rewards.append(total_reward)
            avg_loss = ep_loss_sum / max(ep_loss_count, 1)
            episode_losses.append(avg_loss)
            epsilons.append(epsilon)
            episode_lengths.append(steps)

            metrics = {"episode_reward": total_reward, "epsilon": epsilon}
            if ep_loss_count > 0:
                metrics["loss"] = avg_loss
            await ctx.log_metrics(metrics, step=ep)

            if (ep + 1) % 40 == 0:
                avg_20 = float(np.mean(episode_rewards[-20:]))
                print(
                    f"  ep {ep+1:3d}  reward={total_reward:6.1f}  "
                    f"avg20={avg_20:6.1f}  eps={epsilon:.3f}  loss={avg_loss:.4f}"
                )

        await ctx.log_metric("final_avg_reward", float(np.mean(episode_rewards[-20:])))

    return q_net, episode_rewards, episode_losses, epsilons, episode_lengths


def train_dqn(
    env: gym.Env,
    obs_d: int,
    n_act: int,
    n_episodes: int = 200,
    **kwargs,
) -> tuple[DQN, list[float], list[float], list[float], list[int]]:
    """Sync wrapper for DQN training."""
    return await train_dqn_async(env, obs_d, n_act, n_episodes, **kwargs)


## TASK 3 — Train DQN on CartPole-v1


In [ ]:
print("\n" + "=" * 70)
print("  TASK 3: Train DQN on CartPole-v1")
print("=" * 70)

dqn_model, dqn_rewards, dqn_losses, dqn_epsilons, dqn_lengths = train_dqn(
    cartpole_env, obs_dim, n_actions
)


### Checkpoint 1 ─────────────────────────────────────────────────────


In [ ]:
assert len(dqn_rewards) == 200, "DQN should train for 200 episodes"
assert (
    float(np.mean(dqn_rewards[-20:])) > 50.0
), "DQN should achieve avg reward > 50 in last 20 episodes (random baseline ~20)"
# INTERPRETATION: DQN learns Q(s,a) — the expected total reward from
# taking action a in state s and then acting optimally. The replay buffer
# decorrelates samples; the target network stabilises training. If the
# avg reward is climbing, the Q-network is learning to predict which
# action leads to more pole-balancing time.
print("--- Checkpoint 1 passed --- DQN trained on CartPole\n")


## TASK 4 — Visualise: reward curve, epsilon decay, Q-value heatmap,
episode length progression


In [ ]:
print("=" * 70)
print("  TASK 4: Visualise DQN Agent Behaviour")
print("=" * 70)

viz = ModelVisualizer()


### Plot 1: DQN reward curve with moving average ─────────────────────


In [ ]:
# TODO: Use viz.training_history() with DQN episode rewards and moving average
# Hint: metrics dict with "DQN episode reward": dqn_rewards and
#   "DQN moving avg (20)": moving_average(dqn_rewards, 20)
fig1 = # TODO: viz.training_history(metrics={...}, x_label="Episode", y_label="Reward")
fig1.write_html(str(OUTPUT_DIR / "01_dqn_reward_curve.html"))
print(f"  Saved: {OUTPUT_DIR / '01_dqn_reward_curve.html'}")


### Plot 2: Epsilon decay over training ──────────────────────────────


In [ ]:
# TODO: Plot epsilon values over episodes using viz.training_history()
fig2 = # TODO: viz.training_history(metrics={"Epsilon (exploration rate)": dqn_epsilons}, ...)
fig2.write_html(str(OUTPUT_DIR / "01_dqn_epsilon_decay.html"))
print(f"  Saved: {OUTPUT_DIR / '01_dqn_epsilon_decay.html'}")
# INTERPRETATION: Epsilon starts at 1.0 (100% random) and decays toward
# 0.01. Early episodes are pure exploration — the agent tries everything.
# Later episodes are mostly exploitation — the agent acts on what it learned.
# The curve shape (exponential decay) controls the explore/exploit balance.


### Plot 3: Q-value heatmap over state space ─────────────────────────


In [ ]:
# Visualise what the DQN has learned: for a grid of (cart_position, pole_angle)
# combinations (with velocity=0), what Q-value does it assign to each action?
cart_positions = np.linspace(-2.4, 2.4, 30)
pole_angles = np.linspace(-0.21, 0.21, 30)
q_values_grid = np.zeros((30, 30))

dqn_model.eval()
# TODO: Fill q_values_grid by querying the DQN for each (cart_pos, pole_angle) pair
# Hint: For each (i, cp) and (j, pa), create state [cp, 0.0, pa, 0.0],
#   pass through dqn_model, and store the max Q-value in q_values_grid[j, i]
for i, cp in enumerate(cart_positions):
    for j, pa in enumerate(pole_angles):
        state = torch.tensor([cp, 0.0, pa, 0.0], dtype=torch.float32, device=device)
        with torch.no_grad():
            q_vals = dqn_model(state)
            q_values_grid[j, i] = # TODO: float(q_vals.max().item())

# Use polars for the heatmap data
heatmap_rows = []
for i, cp in enumerate(cart_positions):
    for j, pa in enumerate(pole_angles):
        heatmap_rows.append(
            {
                "cart_position": float(cp),
                "pole_angle": float(pa),
                "max_Q": q_values_grid[j, i],
            }
        )
q_heatmap_df = pl.DataFrame(heatmap_rows)

# Pivot for heatmap visualisation
import plotly.graph_objects as go

# TODO: Create a heatmap figure using go.Heatmap with q_values_grid
# Hint: go.Figure(data=go.Heatmap(z=q_values_grid, x=cart_positions rounded,
#   y=pole_angles rounded, colorscale="Viridis"))
fig3 = # TODO
fig3.update_layout(
    title="DQN Q-Value Heatmap: Cart Position vs Pole Angle (velocities=0)",
    xaxis_title="Cart Position",
    yaxis_title="Pole Angle (radians)",
)
fig3.write_html(str(OUTPUT_DIR / "01_dqn_qvalue_heatmap.html"))
print(f"  Saved: {OUTPUT_DIR / '01_dqn_qvalue_heatmap.html'}")
# INTERPRETATION: The heatmap reveals the DQN's "mental model" of CartPole.
# High Q-values (bright) near the centre (cart centred, pole upright) — the
# agent knows this is a good situation. Low Q-values (dark) at the edges —
# the agent knows recovery is unlikely. This is the learned value landscape.


### Plot 4: Episode length progression ───────────────────────────────


In [ ]:
# TODO: Plot episode lengths and their moving average using viz.training_history()
fig4 = # TODO
fig4.write_html(str(OUTPUT_DIR / "01_dqn_episode_lengths.html"))
print(f"  Saved: {OUTPUT_DIR / '01_dqn_episode_lengths.html'}")
# INTERPRETATION: Episode length IS the reward in CartPole (reward=+1 per
# step). Longer episodes = the agent keeps the pole balanced longer. Early
# episodes are short (random flailing); later episodes approach the 500-step
# maximum (the agent has learned to balance indefinitely).


### Plot 5: DQN loss curve ───────────────────────────────────────────


In [ ]:
# TODO: Plot the Bellman loss curve using viz.training_history()
fig5 = # TODO
fig5.write_html(str(OUTPUT_DIR / "01_dqn_loss_curve.html"))
print(f"  Saved: {OUTPUT_DIR / '01_dqn_loss_curve.html'}")


### Checkpoint 2 ─────────────────────────────────────────────────────


In [ ]:
assert Path(
    OUTPUT_DIR / "01_dqn_reward_curve.html"
).exists(), "Reward curve should be saved"
assert Path(
    OUTPUT_DIR / "01_dqn_qvalue_heatmap.html"
).exists(), "Q-value heatmap should be saved"
assert Path(
    OUTPUT_DIR / "01_dqn_epsilon_decay.html"
).exists(), "Epsilon decay should be saved"
print("--- Checkpoint 2 passed --- all DQN visualisations generated\n")


## TASK 5 — Apply: Inventory Management for a Singapore Retailer
SCENARIO: You're the operations manager at a Singapore supermarket chain
(think FairPrice or Cold Storage). Every day you decide how much stock
to order for a perishable product category (fresh produce). Order too
much -> holding costs and spoilage. Order too little -> empty shelves
and lost sales.
State: (stock_level, demand_forecast, day_of_week)
- stock_level: normalised current inventory [0, 1]
- demand_forecast: predicted demand from the POS system [0, 1]
- day_of_week: cyclical encoding [0, 1] (0=Mon, 1=Sun)
Actions: order_nothing(0), order_small(1), order_medium(2), order_large(3)
Reward: sales_revenue - holding_cost - stockout_penalty - order_cost
Revenue from sales (what you DO sell)
minus cost of holding unsold stock (cold storage, spoilage)
minus penalty for stockouts (lost customers, reputational damage)
minus ordering cost (logistics, supplier handling)


In [ ]:
print("=" * 70)
print("  TASK 5: Apply DQN — Singapore Retail Inventory Management")
print("=" * 70)

from gymnasium import spaces


class RetailInventoryEnv(gym.Env):
    """Inventory management for a Singapore supermarket.

    Models weekly ordering decisions for a perishable product category.
    State (3,): [stock_level, demand_forecast, day_of_week]
    Actions (4): 0=order_nothing, 1=order_small, 2=order_medium, 3=order_large
    Reward: sales - holding_cost - stockout_penalty - order_cost
    Episode: 365 steps (daily decisions for one year).
    """

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(3,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(4)
        self.max_steps = 365
        self.state = None
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array(
            [0.5, 0.4, 0.0],  # start mid-stock, moderate forecast, Monday
            dtype=np.float32,
        )
        self.step_count = 0
        return self.state.copy(), {}

    def step(self, action):
        self.step_count += 1
        stock, forecast, day_norm = self.state

        # TODO: Implement the environment step logic
        # 1. Apply order quantities based on action:
        #    order_qtys = [0.0, 0.08, 0.18, 0.35]
        #    order_costs = [0.0, 0.01, 0.02, 0.04]
        #    stock = min(1.0, stock + order_qtys[action])
        order_qtys = [0.0, 0.08, 0.18, 0.35]
        order_costs = [0.0, 0.01, 0.02, 0.04]
        stock = # TODO: update stock with order quantity

        # 2. Compute demand with seasonal patterns
        # Hint: weekend boost (+0.08 for day_of_week >= 5)
        # Hint: festive boost (+0.12 for CNY weeks 5-6 and Deepavali weeks 43-44)
        day_of_week = int(day_norm * 7) % 7
        weekend_boost = # TODO
        week = self.step_count // 7
        festive_boost = # TODO
        base_demand = 0.15 + weekend_boost + festive_boost
        demand = max(0.0, base_demand + self.np_random.normal(0, 0.04))

        # 3. Fulfil demand and compute spoilage
        # Hint: sold = min(stock, demand), then subtract demand from stock
        # Hint: spoilage = stock * 0.05 (5% daily for perishable goods)
        sold = # TODO
        stockout = # TODO
        stock = # TODO: remaining stock after demand
        spoiled = # TODO: 5% spoilage
        stock = stock - spoiled

        # 4. Compute reward components
        # Hint: sales_revenue = sold * 3.0, holding_cost = stock * 0.3,
        #   stockout_penalty = stockout * 5.0, spoilage_cost = spoiled * 2.0
        sales_revenue = # TODO
        holding_cost = # TODO
        stockout_penalty = # TODO
        spoilage_cost = # TODO

        reward = (
            sales_revenue
            - holding_cost
            - stockout_penalty
            - spoilage_cost
            - order_costs[action]
        )

        # 5. Update state for next step
        day_norm = ((day_of_week + 1) % 7) / 7.0
        forecast = np.clip(base_demand + self.np_random.normal(0, 0.06), 0.0, 1.0)
        self.state = np.array([stock, forecast, day_norm], dtype=np.float32)

        truncated = self.step_count >= self.max_steps
        return self.state.copy(), reward, False, truncated, {}


# Verify environment API
inv_env = RetailInventoryEnv()
obs, info = inv_env.reset(seed=42)
assert obs.shape == (3,), "Inventory env should have 3-D state"
obs2, r, term, trunc, info = inv_env.step(1)
assert isinstance(r, float), "Reward should be float"
print(f"  RetailInventory env: obs={obs.shape}, actions=4, sample_reward={r:.3f}")


### Train DQN on inventory environment ───────────────────────────────


In [ ]:
inv_dqn, inv_rewards, inv_losses, inv_eps, inv_lens = train_dqn(
    inv_env, 3, 4, n_episodes=200, run_name="dqn_retail_inventory"
)


### Fixed-threshold baseline ─────────────────────────────────────────


In [ ]:
# Baseline policy: "order medium when stock < 0.3, order small when < 0.5, else nothing"
def fixed_threshold_policy(state):
    stock = state[0]
    if stock < 0.2:
        return 3  # order large
    elif stock < 0.35:
        return 2  # order medium
    elif stock < 0.5:
        return 1  # order small
    return 0  # order nothing


baseline_returns = evaluate_policy(inv_env, fixed_threshold_policy, n_episodes=30)


def dqn_inventory_policy(state):
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32, device=device)
        return int(inv_dqn(s).argmax().item())


dqn_inv_returns = evaluate_policy(inv_env, dqn_inventory_policy, n_episodes=30)

print(
    f"\n  Fixed-threshold baseline: mean annual reward = {np.mean(baseline_returns):.1f}"
)
print(f"  DQN learned policy:      mean annual reward = {np.mean(dqn_inv_returns):.1f}")

improvement = float(np.mean(dqn_inv_returns)) - float(np.mean(baseline_returns))
pct_improvement = (
    improvement / abs(float(np.mean(baseline_returns))) * 100
    if float(np.mean(baseline_returns)) != 0
    else 0
)
print(f"  Improvement: {improvement:+.1f} ({pct_improvement:+.1f}%)")


### Visualise: DQN vs baseline ───────────────────────────────────────


In [ ]:
# TODO: Create a box plot comparing DQN vs fixed-threshold annual rewards
# Hint: pl.DataFrame with "Policy" and "Annual Reward" columns, then
#   viz.box_plot(comparison_df, "Annual Reward", group_by="Policy")
comparison_df = # TODO
fig_apply = # TODO: viz.box_plot(...)
fig_apply.write_html(str(OUTPUT_DIR / "01_dqn_inventory_comparison.html"))
print(f"  Saved: {OUTPUT_DIR / '01_dqn_inventory_comparison.html'}")


### Visualise: learned policy actions across stock levels ─────────────


In [ ]:
stock_levels = np.linspace(0.0, 1.0, 50)
action_names = ["Nothing", "Small", "Medium", "Large"]
policy_actions = []
inv_dqn.eval()
# TODO: For each stock level, query the DQN with state [sl, 0.3, 0.3]
# and record the chosen action name
for sl in stock_levels:
    state = torch.tensor([sl, 0.3, 0.3], dtype=torch.float32, device=device)
    with torch.no_grad():
        action = # TODO: int(inv_dqn(state).argmax().item())
    policy_actions.append(action_names[action])

policy_df = pl.DataFrame(
    {"Stock Level": stock_levels.tolist(), "Order Action": policy_actions}
)
print("\n  Learned Ordering Policy (demand_forecast=0.3, mid-week):")
for row in policy_df.iter_rows(named=True):
    if row["Stock Level"] in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
        print(f"    Stock={row['Stock Level']:.1f} -> {row['Order Action']}")

# INTERPRETATION: The DQN learns a nuanced ordering policy that considers
# not just current stock level but also the demand forecast. Unlike the
# fixed threshold which uses only stock level, the DQN implicitly learns
# seasonal patterns (order more before CNY/Deepavali weekends) and adjusts
# for the spoilage rate. The annual cost savings translate directly to
# margin improvement for the retailer.

inv_env.close()


### Checkpoint 3 ─────────────────────────────────────────────────────


In [ ]:
assert len(inv_rewards) == 200, "Inventory DQN should train for 200 episodes"
assert Path(OUTPUT_DIR / "01_dqn_inventory_comparison.html").exists()
print("--- Checkpoint 3 passed --- DQN applied to retail inventory\n")


### Register DQN models ──────────────────────────────────────────────


In [ ]:
if has_registry:
    register_rl_model(
        registry,
        "m5_dqn_cartpole",
        dqn_model,
        {
            "avg_reward_last20": float(np.mean(dqn_rewards[-20:])),
            "algorithm": 0.0,
            "episodes_trained": float(len(dqn_rewards)),
        },
    )
    register_rl_model(
        registry,
        "m5_dqn_retail_inventory",
        inv_dqn,
        {
            "avg_reward_last30": float(np.mean(inv_rewards[-30:])),
            "episodes_trained": 200.0,
        },
    )

cartpole_env.close()

# Clean up
await conn.close()


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — DQN")
print("=" * 70)
print(
    """
  [x] Derived the Bellman optimality equation: Q*(s,a) = E[r + gamma * max Q*(s',a')]
  [x] Built DQN from scratch: replay buffer, target network, epsilon-greedy
  [x] Trained DQN on CartPole-v1 — watched it go from random flailing to
      stable pole-balancing
  [x] Visualised the agent's learning:
      - Reward curve: noisy exploration -> smooth convergence
      - Epsilon decay: 100% random -> 1% random
      - Q-value heatmap: the agent's "mental model" of state value
      - Episode lengths: survived longer as it learned
  [x] Applied DQN to Singapore retail inventory management:
      - Built a custom environment with seasonal demand (CNY, Deepavali)
      - DQN learned to outperform a fixed-threshold ordering policy
      - Visualised learned policy vs baseline with annual cost comparison

  KEY INSIGHT:
  DQN learns the VALUE of actions (Q-values), then acts greedily. It's
  like learning to evaluate chess positions before deciding the next move.
  Great for discrete action spaces with clear reward signals.

  Next: In 02_ppo.py, you'll learn PPO — an algorithm that learns the
  POLICY directly (what to do) rather than indirectly through values.
"""
)
